# Build lists of images and texts

In [1]:
import glob
from pathlib import Path

dataset_path = Path(r'C:\Users\Usuario\Downloads\PFC1\DTrOCR\iam_words')

xml_files = sorted(glob.glob(str(dataset_path / 'xml' / '*.xml')))
word_image_files = sorted(glob.glob(str(dataset_path / 'words' / '**' / '*.png'), recursive=True))

print(f"{len(xml_files)} XML files and {len(word_image_files)} word image files")

1539 XML files and 115320 word image files


In [2]:
import tqdm
import multiprocessing as mp
import xml.etree.ElementTree as ET

from PIL import Image
from dataclasses import dataclass

from pathlib import Path

@dataclass
class Word:
    id: str
    file_path: Path
    writer_id: str
    transcription: str

def get_words_from_xml(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    root_id = root.get('id')
    writer_id = root.get('writer-id')
    xml_words = []
    for line in root.findall('handwritten-part')[0].findall('line'):
        for word in line.findall('word'):
            image_file = Path([f for f in word_image_files if f.endswith(word.get('id') + '.png')][0])
            try:
                with Image.open(image_file) as _:
                    xml_words.append(
                        Word(
                            id=root_id,
                            file_path=image_file,
                            writer_id=writer_id,
                            transcription=word.get('text')
                        )
                    )
            except Exception:
                pass
            
    return xml_words

# with mp.Pool(processes=mp.cpu_count()) as pool:
#     words_from_xmls = list(
#         tqdm.tqdm(
#             pool.imap(get_words_from_xml, xml_files), 
#             total=len(xml_files),
#             desc='Building dataset'
#         )
#     )
words_from_xmls = []
for xml_file in tqdm.tqdm(xml_files, desc='Building dataset'):
    words_from_xmls.append(get_words_from_xml(xml_file))

words = [word for words in words_from_xmls for word in words]

Building dataset: 100%|████████████████████████████████████████████████████████████| 1539/1539 [48:35<00:00,  1.89s/it]


In [ ]:
# EJECUTAR EL BLOQUE DE ABAJO SOLO SI SE GUARDO LAS PALABRAS.PKL 

In [2]:
import pickle
import tqdm
import multiprocessing as mp
import xml.etree.ElementTree as ET

from PIL import Image
from dataclasses import dataclass

from pathlib import Path

@dataclass
class Word:
    id: str
    file_path: Path
    writer_id: str
    transcription: str

# Cargamos la lista de objetos 'words.pkl'
with open('dataset_words.pkl', 'rb') as f:
    words = pickle.load(f)

print(f"{len(words)} palabras recuperadas")

115318 palabras recuperadas


# Train test split

In [3]:
with open(dataset_path / 'splits/train.uttlist') as fp:
    train_ids = [line.replace('\n', '') for line in fp.readlines()]

with open(dataset_path / 'splits/test.uttlist') as fp:
    test_ids = [line.replace('\n', '') for line in fp.readlines()]

with open(dataset_path / 'splits/validation.uttlist') as fp:
    validation_ids = [line.replace('\n', '') for line in fp.readlines()]

print(f"Train size: {len(train_ids)}; Validation size: {len(validation_ids)}; Test size: {len(test_ids)}")

Train size: 747; Validation size: 116; Test size: 336


In [4]:
train_word_records = [word for word in words if word.id in train_ids]
validation_word_records = [word for word in words if word.id in validation_ids]
test_word_records = [word for word in words if word.id in test_ids]

print(f'Train size: {len(train_word_records)}; Validation size: {len(validation_word_records)}; Test size: {len(test_word_records)}')

Train size: 55079; Validation size: 8895; Test size: 25920


# Build dataset and dataloader

In [10]:
import sys
import os
# Esto le dice a Python que busque módulos una carpeta más atrás (en la raíz del proyecto)
sys.path.append(os.path.abspath('..'))

from dtrocr.processor import DTrOCRProcessor
from dtrocr.config import DTrOCRConfig

from torch.utils.data import Dataset

class IAMDataset(Dataset):
    def __init__(self, words: list[Word], config: DTrOCRConfig):
        super(IAMDataset, self).__init__()
        self.words = words
        self.processor = DTrOCRProcessor(config, add_eos_token=True, add_bos_token=True)
        
    def __len__(self):
        return len(self.words)
    
    def __getitem__(self, item):
        inputs = self.processor(
            images=Image.open(self.words[item].file_path).convert('RGB'),
            texts=self.words[item].transcription,
            padding='max_length',
            return_tensors="pt",
            return_labels=True,
            input_data_format='channels_first', # Fuerza los canales al principio (3, Alto, Ancho)
        )
        return {
            'pixel_values': inputs.pixel_values[0],
            'input_ids': inputs.input_ids[0],
            'attention_mask': inputs.attention_mask[0],
            'labels': inputs.labels[0]
        }

config = DTrOCRConfig(
    # attn_implementation='flash_attention_2'
)

train_data = IAMDataset(words=train_word_records, config=config)
validation_data = IAMDataset(words=validation_word_records, config=config)
test_data = IAMDataset(words=test_word_records, config=config)
print("okey")

okey


In [11]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_data, batch_size=8, shuffle=True, num_workers=0)
validation_dataloader = DataLoader(validation_data, batch_size=8, shuffle=False, num_workers=0)
test_dataloader = DataLoader(test_data, batch_size=8, shuffle=False, num_workers=0)
print("Okey")

Okey


# Model

In [12]:
import torch
torch.set_float32_matmul_precision('high')

from dtrocr.model import DTrOCRLMHeadModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Entrenando en: {device}")

# model = DTrOCRLMHeadModel(config)
# model = torch.compile(model)
# model.to(device=0)

model = DTrOCRLMHeadModel(config)
model.to(device)
print("okey")

Entrenando en: cuda
okey


In [13]:
import pickle

# Guarda la lista de objetos 'words' en un archivo local
with open('dataset_words.pkl', 'wb') as f:
    pickle.dump(words, f)

print("words guardado!")

words guardado!


# Training

In [ ]:
import logging
import sys
import os

# Crear el logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("run.log", mode='a', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)    ]
)
logging.info("Iniciando el proceso de entrenamiento DTrOCR...")

# training
from typing import Tuple

def evaluate_model(model: torch.nn.Module, dataloader: DataLoader) -> Tuple[float, float]:
    # set model to evaluation mode
    model.eval()
    
    losses, accuracies = [], []
    with torch.no_grad():
        for inputs in tqdm.tqdm(dataloader, total=len(dataloader), desc=f'Evaluating test set'):
            # inputs = send_inputs_to_device(inputs, device=0)
            inputs = send_inputs_to_device(inputs, device=device)
            outputs = model(**inputs)
            
            losses.append(outputs.loss.item())
            accuracies.append(outputs.accuracy.item())
    
    loss = sum(losses) / len(losses)
    accuracy = sum(accuracies) / len(accuracies)
    
    # set model back to training mode
    model.train()
    
    return loss, accuracy

def send_inputs_to_device(dictionary, device):
    return {key: value.to(device=device) if isinstance(value, torch.Tensor) else value for key, value in dictionary.items()}

use_amp = True
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
optimiser = torch.optim.Adam(params=model.parameters(), lr=1e-4)

EPOCHS = 50
train_losses, train_accuracies = [], []
validation_losses, validation_accuracies = [], []
for epoch in range(EPOCHS):
    epoch_losses, epoch_accuracies = [], []
    for inputs in tqdm.tqdm(train_dataloader, total=len(train_dataloader), desc=f'Epoch {epoch + 1}'):
        
        # set gradients to zero
        optimiser.zero_grad()
        
        # send inputs to same device as model
        # inputs = send_inputs_to_device(inputs, device=0)
        inputs = send_inputs_to_device(inputs, device=device)
        
        # forward pass
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            outputs = model(**inputs)
        
        # calculate gradients
        scaler.scale(outputs.loss).backward()
        
        # update weights
        scaler.step(optimiser)
        scaler.update()
        
        epoch_losses.append(outputs.loss.item())
        epoch_accuracies.append(outputs.accuracy.item())
        
    # store loss and metrics
    train_losses.append(sum(epoch_losses) / len(epoch_losses))
    train_accuracies.append(sum(epoch_accuracies) / len(epoch_accuracies))
    
    # tests loss and accuracy
    validation_loss, validation_accuracy = evaluate_model(model, validation_dataloader)
    validation_losses.append(validation_loss)
    validation_accuracies.append(validation_accuracy)
                    
    # print(f"Epoch: {epoch + 1} - Train loss: {train_losses[-1]}, Train accuracy: {train_accuracies[-1]}, Validation loss: {validation_losses[-1]}, Validation accuracy: {validation_accuracies[-1]}")

    # # guardar pesos
    # import os
    # os.makedirs('output', exist_ok=True)
    # # Guardar el modelo al final de cada época
    # save_path = f"output/dtrocr_epoch_{epoch + 1}.pth"
    # torch.save(model.state_dict(), save_path)
    # print(f"Modelo guardado en: {save_path}")
    # Registrar métricas de la época en el log
    
    logging.info(
        f"Epoch: {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_losses[-1]:.4f} | Train Acc: {train_accuracies[-1]:.4f} | "
        f"Val Loss: {validation_losses[-1]:.4f} | Val Acc: {validation_accuracies[-1]:.4f}"
    )

    # Guardar pesos
    os.makedirs('output', exist_ok=True)
    save_path = f"output/dtrocr_epoch_{epoch + 1}.pth"
    torch.save(model.state_dict(), save_path)
    logging.info(f"Modelo guardado exitosamente en: {save_path}")
print("okey")

2026-05-27 02:16:18,888 - INFO - Iniciando el proceso de entrenamiento DTrOCR...


C:\Users\Usuario\AppData\Local\Temp\ipykernel_14156\2887170258.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
Evaluating test set: 100%|██████████████████████████████████████████████████████████| 1112/1112 [01:31<00:00, 12.14it/s]

2026-05-27 02:39:13,324 - INFO - Epoch: 1/50 | Train Loss: 3.3996 | Train Acc: 0.5413 | Val Loss: 2.9374 | Val Acc: 0.5968


2026-05-27 02:39:14,127 - INFO - Modelo guardado exitosamente en: output/dtrocr_epoch_1.pth


Evaluating test set: 100%|██████████████████████████████████████████████████████████| 1112/1112 [01:26<00:00, 12.78it/s]

...

2026-05-27 04:29:13,550 - INFO - Modelo guardado exitosamente en: output/dtrocr_epoch_6.pth


Evaluating test set: 100%|██████████████████████████████████████████████████████████| 1112/1112 [01:27<00:00, 12.78it/s]

2026-05-27 04:51:21,743 - INFO - Epoch: 7/50 | Train Loss: 0.5394 | Train Acc: 0.8787 | Val Loss: 2.2485 | Val Acc: 0.7003


2026-05-27 04:51:22,681 - INFO - Modelo guardado exitosamente en: output/dtrocr_epoch_7.pth


Epoch 8:   9%|██████▍                                                                | 621/6885 [01:50<18:26,  5.66it/s]

# Test

In [ ]:
from dtrocr.model import DTrOCRLMHeadModel
from dtrocr.config import DTrOCRConfig
from dtrocr.processor import DTrOCRProcessor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = DTrOCRLMHeadModel(DTrOCRConfig())
model.eval()
# model.to('cpu')
model.to(device)
# test_processor = DTrOCRProcessor(DTrOCRConfig())
test_processor = DTrOCRProcessor(DTrOCRConfig(), add_bos_token=True, add_eos_token=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import tqdm
from jiwer import cer

ground_truths = []
predictions = []

print("Generando predicciones para el cálculo del CER...")
for test_word_record in tqdm.tqdm(test_word_records):
    image_file = test_word_record.file_path
    real_text = test_word_record.transcription
    
    # Prepara la imagen
    image = Image.open(image_file).convert('RGB')
    
    inputs = test_processor(
        images=image, 
        texts=test_processor.tokeniser.bos_token,
        return_tensors='pt',
        input_data_format='channels_first'
    )
    
    # inputs = {k: v.to(model.device) if hasattr(v, 'to') else v for k, v in inputs.items()}
    # inputs = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs.items()}

    if inputs.pixel_values is not None:
        inputs.pixel_values = inputs.pixel_values.to(device)
    if inputs.input_ids is not None:
        inputs.input_ids = inputs.input_ids.to(device)
    if inputs.attention_mask is not None:
        inputs.attention_mask = inputs.attention_mask.to(device)
    
    # Generar texto
    model_output = model.generate(
        inputs, 
        test_processor, 
        num_beams=3, 
        use_cache=True
    )
    
    predicted_text = test_processor.tokeniser.decode(model_output[0], skip_special_tokens=True)
    
    ground_truths.append(real_text)
    predictions.append(predicted_text)

# Calcular el CER
error_rate = cer(ground_truths, predictions)

# print(f"\n======================================")
# print(f"Total de palabras evaluadas: {len(ground_truths)}")
# print(f"Character Error Rate (CER): {error_rate * 100:.2f}%")
# print(f"======================================")

logging.info("======================================")
logging.info(f"Evaluación Final Completada")
logging.info(f"Total de palabras evaluadas: {len(ground_truths)}")
logging.info(f"Character Error Rate (CER): {error_rate * 100:.2f}%")
logging.info("======================================")

print("\nMuestra de predicciones:")
for i in range(10):
    print(f"Real: '{ground_truths[i]}' | Predicción: '{predictions[i]}'")

# from PIL import Image

# import numpy as np
# import matplotlib.pyplot as plt

# from PIL import Image

# for test_word_record in test_word_records[:50]:
#     image_file = test_word_record.file_path
#     image = Image.open(image_file).convert('RGB')
    
#     inputs = test_processor(
#         images=image, 
#         texts=test_processor.tokeniser.bos_token,
#         return_tensors='pt'
#     )
    
#     model_output = model.generate(
#         inputs, 
#         test_processor, 
#         num_beams=3
#     )
    
#     predicted_text = test_processor.tokeniser.decode(model_output[0], skip_special_tokens=True)
    
#     plt.figure(figsize=(10, 5))
#     plt.title(predicted_text, fontsize=24)
#     plt.imshow(np.array(image, dtype=np.uint8))
#     plt.xticks([]), plt.yticks([])
#     plt.show()